In [10]:
import torch
import torch.nn.functional as F

torch.manual_seed(7)

# ------------------------------------------------------------
# 1. Tiny English -> German dataset
# ------------------------------------------------------------

pairs = [
    ("i like cars",      "ich mag autos"),
    ("i like cats",      "ich mag katzen"),
    ("you like cars",    "du magst autos"),
    ("you like cats",    "du magst katzen"),
    ("i see cars",       "ich sehe autos"),
    ("you see cats",     "du siehst katzen"),

    # Important examples:
    # English has 3 words, German has 4 words
    ("i am driving",     "ich fahre gerade auto"),
    ("you are driving",  "du faehrst gerade auto"),
]

BOS = "<BOS>"   # beginning of sentence
EOS = "<EOS>"   # end of sentence

# ------------------------------------------------------------
# 2. Build separate English and German vocabularies
# ------------------------------------------------------------

english_words = sorted({
    word
    for english_sentence, _ in pairs
    for word in english_sentence.split()
})

german_words = [BOS, EOS] + sorted({
    word
    for _, german_sentence in pairs
    for word in german_sentence.split()
})

eng_stoi = {word: i for i, word in enumerate(english_words)}
eng_itos = {i: word for word, i in eng_stoi.items()}

de_stoi = {word: i for i, word in enumerate(german_words)}
de_itos = {i: word for word, i in de_stoi.items()}

print("English vocabulary:")
print(eng_stoi)

print("\nGerman vocabulary:")
print(de_stoi)


def encode_english_sentence(sentence):
    return torch.tensor(
        [eng_stoi[word] for word in sentence.split()],
        dtype=torch.long
    )


def encode_german_target(sentence):
    """
    German target includes EOS.

    Example:
        "ich mag autos"

    becomes:
        ["ich", "mag", "autos", "<EOS>"]

    The EOS token teaches the decoder when to stop.
    """
    return torch.tensor(
        [de_stoi[word] for word in sentence.split()] + [de_stoi[EOS]],
        dtype=torch.long
    )


training_data = [
    (
        encode_english_sentence(english_sentence),
        encode_german_target(german_sentence),
        english_sentence,
        german_sentence
    )
    for english_sentence, german_sentence in pairs
]

# ------------------------------------------------------------
# 3. Manual parameter creation
# ------------------------------------------------------------

def make_param(*shape, scale=0.1):
    parameter = torch.randn(*shape) * scale

    # detach().requires_grad_() makes sure it is a leaf tensor
    parameter = parameter.detach().requires_grad_()

    return parameter


embedding_dim = 16
hidden_size = 32

english_vocab_size = len(eng_stoi)
german_vocab_size = len(de_stoi)

# ------------------------------------------------------------
# 4. Two different embedding tables
# ------------------------------------------------------------

# English embedding: used by the encoder
E_eng = make_param(english_vocab_size, embedding_dim)

# German embedding: used by the decoder
E_de = make_param(german_vocab_size, embedding_dim)

# ------------------------------------------------------------
# 5. Encoder RNN parameters
# ------------------------------------------------------------

W_enc_xh = make_param(embedding_dim, hidden_size)
W_enc_hh = make_param(hidden_size, hidden_size)
b_enc_h = torch.zeros(hidden_size, requires_grad=True)

# ------------------------------------------------------------
# 6. Decoder RNN parameters
# ------------------------------------------------------------

W_dec_xh = make_param(embedding_dim, hidden_size)
W_dec_hh = make_param(hidden_size, hidden_size)
b_dec_h = torch.zeros(hidden_size, requires_grad=True)

# ------------------------------------------------------------
# 7. Output layer with attention
# ------------------------------------------------------------

# Without attention:
#   hidden_size -> german_vocab_size
#
# With attention:
#   [decoder_hidden, attention_context] -> german_vocab_size
#
# decoder_hidden shape:    [hidden_size]
# attention_context shape: [hidden_size]
# concatenated shape:      [2 * hidden_size]

W_out = make_param(2 * hidden_size, german_vocab_size)
b_out = torch.zeros(german_vocab_size, requires_grad=True)

parameters = [
    E_eng,
    E_de,

    W_enc_xh,
    W_enc_hh,
    b_enc_h,

    W_dec_xh,
    W_dec_hh,
    b_dec_h,

    W_out,
    b_out,
]

English vocabulary:
{'am': 0, 'are': 1, 'cars': 2, 'cats': 3, 'driving': 4, 'i': 5, 'like': 6, 'see': 7, 'you': 8}

German vocabulary:
{'<BOS>': 0, '<EOS>': 1, 'auto': 2, 'autos': 3, 'du': 4, 'faehrst': 5, 'fahre': 6, 'gerade': 7, 'ich': 8, 'katzen': 9, 'mag': 10, 'magst': 11, 'sehe': 12, 'siehst': 13}


In [11]:
#The encoder reads the English sentence word by word.
def encoder(src_ids):
    """
    src_ids shape:
        [source_sentence_length]

    Example:
        "i am driving"

    src_ids length:
        3

    Returns:
        final hidden state h
    """

    h = torch.zeros(hidden_size)

    hidden_states = []

    for token_id in src_ids:

        # English word embedding
        #
        # token_id is an integer
        # E_eng[token_id] is one vector of shape [embedding_dim]
        x_t = E_eng[token_id]

        # Manual RNN equation:
        #
        # h_t = tanh(x_t W_xh + h_{t-1} W_hh + b)
        h = torch.tanh(
            x_t @ W_enc_xh +
            h   @ W_enc_hh +
            b_enc_h
        )

        hidden_states.append(h)

    hidden_states = torch.stack(hidden_states)

    # The final h is the context vector
    return h, hidden_states

In [12]:
def dot_attention(decoder_hidden, encoder_hidden_states):
    """
    decoder_hidden shape:
        [hidden_size]

    encoder_hidden_states shape:
        [source_length, hidden_size]

    Returns:
        context:
            [hidden_size]

        attention_weights:
            [source_length]
    """

    # --------------------------------------------------------
    # 1. Similarity scores
    # --------------------------------------------------------
    #
    # encoder_hidden_states: [source_length, hidden_size]
    # decoder_hidden:        [hidden_size]
    #
    # scores:                [source_length]
    #
    # Each score says:
    #   how relevant is this English position
    #   for the current German decoder step?
    scores = encoder_hidden_states @ decoder_hidden

    # --------------------------------------------------------
    # 2. Convert scores into probabilities
    # --------------------------------------------------------
    attention_weights = F.softmax(scores, dim=0)

    # --------------------------------------------------------
    # 3. Weighted sum of encoder states
    # --------------------------------------------------------
    #
    # attention_weights:      [source_length]
    # encoder_hidden_states:  [source_length, hidden_size]
    #
    # context:                [hidden_size]
    context = attention_weights @ encoder_hidden_states

    return context, attention_weights

In [13]:
#The decoder produces German words.
#During training, the decoder is fed the correct previous German word. This is called teacher forcing.
def decoder_train_with_attention(initial_hidden, encoder_hidden_states, tgt_ids):
    """
    context:
        final encoder hidden state

    tgt_ids:
        German target token ids, including EOS.

    Example:
        "ich fahre gerade auto <EOS>"

    Returns:
        logits_over_time:
            [target_length, german_vocab_size]

        attention_over_time:
            [target_length, source_length]
    """

    h = initial_hidden

    # First decoder input is always BOS
    prev_token_id = torch.tensor(de_stoi[BOS], dtype=torch.long)

    logits_over_time = []
    attention_over_time = []

    for target_token_id in tgt_ids:

        # German embedding of previous German token
        x_t = E_de[prev_token_id]

        # Manual decoder RNN equation - 1. Decoder RNN step
        h = torch.tanh(
            x_t @ W_dec_xh +
            h   @ W_dec_hh +
            b_dec_h
        )

        # ----------------------------------------------------
        # 2. Attention over all encoder states
        # ----------------------------------------------------

        context_t, attention_weights = dot_attention(
            decoder_hidden=h,
            encoder_hidden_states=encoder_hidden_states
        )        

        # ----------------------------------------------------
        # 3. Combine decoder hidden state and attention context
        # ----------------------------------------------------

        combined = torch.cat([h, context_t], dim=0)        

        # 4. Predict next German word
        # Fully connected output layer:
        # hidden state -> German word scores
        logits_t = combined @ W_out + b_out

        logits_over_time.append(logits_t)
        attention_over_time.append(attention_weights)

        # Teacher forcing:
        # Feed the correct previous word into the next decoder step.
        prev_token_id = target_token_id

    return torch.stack(logits_over_time), torch.stack(attention_over_time)

In [19]:
#Training loop - This version computes one averaged loss per epoch and calls backward() once per epoch. That is much faster than doing one backward pass per sentence.
learning_rate = 0.15
epochs = 1500

for epoch in range(epochs):

    losses = []

    for src_ids, tgt_ids, english_sentence, german_sentence in training_data:

        # Encoder
        final_hidden, encoder_hidden_states = encoder(src_ids)

        # Decoder
        logits, attention_weights = decoder_train_with_attention(initial_hidden=final_hidden, encoder_hidden_states=encoder_hidden_states, tgt_ids=tgt_ids)

        # logits shape:
        #   [target_length, german_vocab_size]
        #
        # tgt_ids shape:
        #   [target_length]
        loss = F.cross_entropy(logits, tgt_ids)
        losses.append(loss)

    total_loss = torch.stack(losses).mean()

    # Clear old gradients
    for p in parameters:
        p.grad = None

    # Backpropagation through encoder and decoder time steps
    total_loss.backward()

    # Gradient descent
    with torch.no_grad():
        for p in parameters:
            p -= learning_rate * p.grad    

    if epoch % 300 == 0:
        average_loss = total_loss / len(pairs)
        print(f"epoch {epoch:4d} | average loss {average_loss:.4f}")

epoch    0 | average loss 0.0012
epoch  300 | average loss 0.0008
epoch  600 | average loss 0.0006
epoch  900 | average loss 0.0004
epoch 1200 | average loss 0.0004


In [20]:
#Inference / translation
#During inference, we do not know the correct previous German word.
#So the decoder feeds its own previous prediction back into itself:

# <BOS> ─► predicts "ich"
# "ich" ─► predicts "fahre"
# "fahre" ─► predicts "gerade"
# "gerade" ─► predicts "auto"
# "auto" ─► predicts "<EOS>" and stops

def translate(english_sentence, max_steps=10):

    with torch.no_grad():

        src_ids = encode_english_sentence(english_sentence)

        # Encode English sentence
        final_hidden, encoder_hidden_states = encoder(src_ids)

        # Start decoder from encoder context
        h = final_hidden

        prev_token_id = torch.tensor(de_stoi[BOS], dtype=torch.long)

        output_words = []
        attention_rows = []

        for step in range(max_steps):

            # German embedding of previous German word
            x_t = E_de[prev_token_id]

            h = torch.tanh(
                x_t @ W_dec_xh +
                h   @ W_dec_hh +
                b_dec_h
            )

            context_t, attention_weights = dot_attention(
                decoder_hidden=h,
                encoder_hidden_states=encoder_hidden_states
            )

            combined = torch.cat([h, context_t], dim=0)            

            logits_t = combined @ W_out + b_out

            next_token_id = int(torch.argmax(logits_t))

            attention_rows.append(attention_weights)

            if next_token_id == de_stoi[EOS]:
                break

            output_words.append(de_itos[next_token_id])

            prev_token_id = torch.tensor(next_token_id, dtype=torch.long)

        if len(attention_rows) > 0:
            attention_matrix = torch.stack(attention_rows)
        else:
            attention_matrix = torch.empty(0)

        return " ".join(output_words), attention_matrix

In [21]:
#Test
#Because this is a tiny toy dataset, the model is mostly memorizing. That is fine here because the goal is to understand the architecture.
print("\nTranslations:")

for _, _, english_sentence, german_sentence in training_data:
    prediction, attention_matrix = translate(english_sentence)

    print(
        f"{english_sentence:18s} -> "
        f"{prediction:28s} | target: {german_sentence}"
    )


Translations:
i like cars        -> ich mag autos                | target: ich mag autos
i like cats        -> ich mag katzen               | target: ich mag katzen
you like cars      -> du magst autos               | target: du magst autos
you like cats      -> du magst katzen              | target: du magst katzen
i see cars         -> ich sehe autos               | target: ich sehe autos
you see cats       -> du siehst katzen             | target: du siehst katzen
i am driving       -> ich fahre gerade auto        | target: ich fahre gerade auto
you are driving    -> du faehrst gerade auto       | target: du faehrst gerade auto


In [22]:
def print_attention(english_sentence):

    prediction, attention_matrix = translate(english_sentence)

    src_words = english_sentence.split()

    # attention has one row per predicted word, including the EOS step.
    target_words = prediction.split() + [EOS]

    print("\nEnglish sentence:")
    print(english_sentence)

    print("\nPredicted German sentence:")
    print(prediction)

    print("\nAttention matrix:")
    print()

    header = f"{'target/src':>12s}"
    for src_word in src_words:
        header += f"{src_word:>12s}"
    print(header)

    for target_word, weights in zip(target_words, attention_matrix):

        row = f"{target_word:>12s}"

        for weight in weights:
            row += f"{float(weight):12.3f}"

        print(row)

In [23]:
print_attention("i am driving")


English sentence:
i am driving

Predicted German sentence:
ich fahre gerade auto

Attention matrix:

  target/src           i          am     driving
         ich       0.986       0.014       0.000
       fahre       0.968       0.023       0.009
      gerade       0.030       0.967       0.002
        auto       0.003       0.001       0.996
       <EOS>       0.053       0.006       0.940
